In [1]:
# ------------------------------------------------------------
# 02 — LTV MODELING
# ------------------------------------------------------------
# Goal:
# Estimate customer lifetime value using historical
# purchasing behavior.
#
# We will first understand the customer-level data,
# then construct features, and only then build the model.
# ------------------------------------------------------------

import pandas as pd
import numpy as np

from pathlib import Path

# Set the location of the raw datasets
DATA_DIR = Path("../data/raw")

# Load the datasets needed for LTV analysis
orders = pd.read_csv(DATA_DIR / "olist_orders_dataset.csv")
items = pd.read_csv(DATA_DIR / "olist_order_items_dataset.csv")
customers = pd.read_csv(DATA_DIR / "olist_customers_dataset.csv")
payments = pd.read_csv(DATA_DIR / "olist_order_payments_dataset.csv")

# Convert order purchase timestamp to datetime
orders["purchase"] = pd.to_datetime(orders["order_purchase_timestamp"])

# Check that the data loaded correctly
display(orders.head())
display(items.head())
display(customers.head())
display(payments.head())

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,purchase
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,2017-10-02 10:56:33
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,2018-07-24 20:41:37
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,2018-08-08 08:38:49
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,2017-11-18 19:28:06
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,2018-02-13 21:18:39


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [2]:
# ------------------------------------------------------------
# LOAD RAW DATASETS
# ------------------------------------------------------------
# We load the original CSV files from the raw data folder.
# These raw files will NOT be modified directly.
# ------------------------------------------------------------

customers = pd.read_csv(DATA_DIR / "olist_customers_dataset.csv")
geolocation = pd.read_csv(DATA_DIR / "olist_geolocation_dataset.csv")
orders = pd.read_csv(DATA_DIR / "olist_orders_dataset.csv")
items = pd.read_csv(DATA_DIR / "olist_order_items_dataset.csv")
payments = pd.read_csv(DATA_DIR / "olist_order_payments_dataset.csv")
reviews = pd.read_csv(DATA_DIR / "olist_order_reviews_dataset.csv")
products = pd.read_csv(DATA_DIR / "olist_products_dataset.csv")
sellers = pd.read_csv(DATA_DIR / "olist_sellers_dataset.csv")
translation = pd.read_csv(DATA_DIR / "product_category_name_translation.csv")

print("Datasets loaded successfully.")

Datasets loaded successfully.


In [3]:
# ------------------------------------------------------------
# CONVERT DATE COLUMNS TO DATETIME
# ------------------------------------------------------------
# The raw CSVs store dates as strings.
# We convert them to datetime so we can:
# - calculate time differences
# - compare dates
# - create time-based features
# ------------------------------------------------------------

# Orders
orders["order_purchase_timestamp"] = pd.to_datetime(
    orders["order_purchase_timestamp"]
)

orders["order_approved_at"] = pd.to_datetime(
    orders["order_approved_at"]
)

orders["order_delivered_carrier_date"] = pd.to_datetime(
    orders["order_delivered_carrier_date"]
)

orders["order_delivered_customer_date"] = pd.to_datetime(
    orders["order_delivered_customer_date"]
)

orders["order_estimated_delivery_date"] = pd.to_datetime(
    orders["order_estimated_delivery_date"]
)

# Order Items
items["shipping_limit_date"] = pd.to_datetime(
    items["shipping_limit_date"]
)

# Reviews
reviews["review_creation_date"] = pd.to_datetime(
    reviews["review_creation_date"]
)

reviews["review_answer_timestamp"] = pd.to_datetime(
    reviews["review_answer_timestamp"]
)

print("Date conversion completed.")

Date conversion completed.


In [4]:
# ------------------------------------------------------------
# VERIFY DATE TYPES
# ------------------------------------------------------------
# Check that all date columns are now stored as datetime.
# ------------------------------------------------------------

print("ORDERS")
display(orders[[
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]].dtypes)

print("\nORDER ITEMS")
display(items[[
    "shipping_limit_date"
]].dtypes)

print("\nREVIEWS")
display(reviews[[
    "review_creation_date",
    "review_answer_timestamp"
]].dtypes)

ORDERS


order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object


ORDER ITEMS


shipping_limit_date    datetime64[us]
dtype: object


REVIEWS


review_creation_date       datetime64[us]
review_answer_timestamp    datetime64[us]
dtype: object

In [5]:
# ------------------------------------------------------------
# CHECK PRODUCT MISSING-VALUE PATTERN
# ------------------------------------------------------------
# We want to know whether the 610 missing values occur
# in the same products or in different products.
# ------------------------------------------------------------

pm = products.isna().sum()

display(pm[pm > 0])

product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

In [6]:
# ------------------------------------------------------------
# CHECK HOW MANY PRODUCTS HAVE ANY MISSING VALUE
# ------------------------------------------------------------
# This tells us how many unique product records are affected.
# ------------------------------------------------------------

pm_rows = products.isna().any(axis=1).sum()

print("Products with missing values:", pm_rows)

Products with missing values: 611


In [7]:
# ------------------------------------------------------------
# SHOW THE AFFECTED PRODUCTS
# ------------------------------------------------------------
# This lets us inspect exactly what is missing.
# ------------------------------------------------------------

pm_data = products[products.isna().any(axis=1)]

display(pm_data)

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
105,a41e356c76fab66334f36de622ecbd3a,NaN,NaN,NaN,NaN,650.0,17.0,14.0,12.0
128,d8dee61c2034d6d075997acef1870e9b,NaN,NaN,NaN,NaN,300.0,16.0,7.0,20.0
145,56139431d72cd51f19eb9f7dae4d1617,NaN,NaN,NaN,NaN,200.0,20.0,20.0,20.0
154,46b48281eb6d663ced748f324108c733,NaN,NaN,NaN,NaN,18500.0,41.0,30.0,41.0
197,5fb61f482620cb672f5e586bb132eae9,NaN,NaN,NaN,NaN,300.0,35.0,7.0,12.0
...,...,...,...,...,...,...,...,...,...
32515,b0a0c5dd78e644373b199380612c350a,NaN,NaN,NaN,NaN,1800.0,30.0,20.0,70.0
32589,10dbe0fbaa2c505123c17fdc34a63c56,NaN,NaN,NaN,NaN,800.0,30.0,10.0,23.0
32616,bd2ada37b58ae94cc838b9c0569fecd8,NaN,NaN,NaN,NaN,200.0,21.0,8.0,16.0
32772,fa51e914046aab32764c41356b9d4ea4,NaN,NaN,NaN,NaN,1300.0,45.0,16.0,45.0


In [8]:
# ------------------------------------------------------------
# FIND PRODUCTS WITH A DIFFERENT MISSING-VALUE PATTERN
# ------------------------------------------------------------
# Most missing products have the same 4 fields missing.
# We find the product(s) that don't follow this pattern.
# ------------------------------------------------------------

cols = [
    "product_category_name",
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty"
]

# Count missing values in these 4 main product fields
products["missing_count"] = products[cols].isna().sum(axis=1)

# Show products where the pattern is different
pm_diff = products[
    products["missing_count"] != 4
]

display(pm_diff)

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,missing_count
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0,0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0,0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0,0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0,0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0,0
...,...,...,...,...,...,...,...,...,...,...
32946,a0b7d5a992ccda646f2d34e418fff5a0,moveis_decoracao,45.0,67.0,2.0,12300.0,40.0,40.0,40.0,0
32947,bf4538d88321d0fd4412a93c974510e6,construcao_ferramentas_iluminacao,41.0,971.0,1.0,1700.0,16.0,19.0,16.0,0
32948,9a7c6041fa9592d9d9ef6cfe62a71f8c,cama_mesa_banho,50.0,799.0,1.0,1400.0,27.0,7.0,27.0,0
32949,83808703fc0706a22e264b9d75f04a2e,informatica_acessorios,60.0,156.0,2.0,700.0,31.0,13.0,20.0,0


In [9]:
# ------------------------------------------------------------
# CHECK THE TWO MISSING-PHYSICAL-DATA PRODUCTS
# ------------------------------------------------------------

physical = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

pm_physical = products[
    products[physical].isna().any(axis=1)
]

display(pm_physical)

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,missing_count
8578,09ff539a621711667c43eba6a3bd8466,bebes,60.0,865.0,3.0,NaN,NaN,NaN,NaN,0
18851,5eb564652db742ff8f28759cd8d2652a,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4


In [10]:
# ------------------------------------------------------------
# REMOVE TEMPORARY AUDIT COLUMN
# ------------------------------------------------------------
# missing_count was only used to investigate the missing
# value pattern. It is not part of the original data.
# ------------------------------------------------------------

products.drop(columns="missing_count", inplace=True)

In [11]:
# ------------------------------------------------------------
# VERIFY PRODUCT TABLE
# ------------------------------------------------------------

display(products.head())
print("Products:", len(products))
print("Columns:", len(products.columns))

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


Products: 32951
Columns: 9


In [12]:
# ------------------------------------------------------------
# CHECK REVIEW MISSING VALUES
# ------------------------------------------------------------
# A review can have a score even when the customer does not
# write a title or comment.
# We first check how many reviews have each type of missing data.
# ------------------------------------------------------------

rm = reviews[[
    "review_score",
    "review_comment_title",
    "review_comment_message"
]].isna().sum()

display(rm.to_frame("missing"))

,missing
review_score,0
review_comment_title,87656
review_comment_message,58247


In [13]:
# ------------------------------------------------------------
# CHECK REVIEW COMMENT PATTERNS
# ------------------------------------------------------------
# We check whether reviews have:
# 1. Both title and message
# 2. Title only
# 3. Message only
# 4. Neither title nor message
# ------------------------------------------------------------

rpat = reviews.assign(
    title_missing=reviews["review_comment_title"].isna(),
    message_missing=reviews["review_comment_message"].isna()
)

rpat = (
    rpat
    .groupby(["title_missing", "message_missing"])
    .size()
    .reset_index(name="reviews")
)

display(rpat)

,title_missing,message_missing,reviews
0,False,False,9839
1,False,True,1729
2,True,False,31138
3,True,True,56518


In [14]:
# ------------------------------------------------------------
# CHECK INVALID PAYMENT RECORDS
# ------------------------------------------------------------
# We identified zero-value payments and invalid installment
# values during the data audit.
# We inspect them before deciding whether to remove them.
# ------------------------------------------------------------

p0 = payments[
    payments["payment_value"] <= 0
]

pinst = payments[
    payments["payment_installments"] <= 0
]

display(p0)
display(pinst)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
19922,8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.0
36822,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.0
43744,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.0
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
62674,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.0
77885,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.0
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0
100766,b23878b3e8eb4d25a158f57d96331b18,4,voucher,1,0.0


,order_id,payment_sequential,payment_type,payment_installments,payment_value
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94


In [15]:
# ------------------------------------------------------------
# CHECK STATUS OF NOT_DEFINED PAYMENTS
# ------------------------------------------------------------
# We join the payment records with orders to understand
# what happened to these three payments.
# ------------------------------------------------------------

pnd = payments[
    payments["payment_type"] == "not_defined"
].merge(
    orders[["order_id", "order_status"]],
    on="order_id",
    how="left"
)

display(pnd)

,order_id,payment_sequential,payment_type,payment_installments,payment_value,order_status
0,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0,canceled
1,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0,canceled
2,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0,canceled


In [16]:
# ------------------------------------------------------------
# FIND ORDERS WITHOUT PAYMENT RECORDS
# ------------------------------------------------------------
# We identify orders that exist in the orders table but have
# no corresponding record in the payments table.
# ------------------------------------------------------------

op = orders[
    ~orders["order_id"].isin(payments["order_id"])
]

display(op)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
30710,bfbd0f9bdef84302105ad712db648a6c,86dc2ffce2dfff336de2f386a786e574,delivered,2016-09-15 12:16:38,2016-09-15 12:16:38,2016-11-07 17:11:53,2016-11-09 07:47:38,2016-10-04


In [17]:
# ------------------------------------------------------------
# INVESTIGATE ORDER WITHOUT PAYMENT
# ------------------------------------------------------------
# We inspect the order, its items, and its review.
# This helps us understand whether the missing payment is
# likely an incomplete payment record rather than a bad order.
# ------------------------------------------------------------

o = orders[orders["order_id"] == "bfbd0f9bdef84302105ad712db648a6c"]

i = items[
    items["order_id"] == "bfbd0f9bdef84302105ad712db648a6c"
]

r = reviews[
    reviews["order_id"] == "bfbd0f9bdef84302105ad712db648a6c"
]

display(o)
display(i)
display(r)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
30710,bfbd0f9bdef84302105ad712db648a6c,86dc2ffce2dfff336de2f386a786e574,delivered,2016-09-15 12:16:38,2016-09-15 12:16:38,2016-11-07 17:11:53,2016-11-09 07:47:38,2016-10-04


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
84389,bfbd0f9bdef84302105ad712db648a6c,1,5a6b04657a4c5ee34285d1e4619a96b4,ecccfa2bb93b34a3bf033cc5d1dcdc69,2016-09-19 23:11:33,44.99,2.83
84390,bfbd0f9bdef84302105ad712db648a6c,2,5a6b04657a4c5ee34285d1e4619a96b4,ecccfa2bb93b34a3bf033cc5d1dcdc69,2016-09-19 23:11:33,44.99,2.83
84391,bfbd0f9bdef84302105ad712db648a6c,3,5a6b04657a4c5ee34285d1e4619a96b4,ecccfa2bb93b34a3bf033cc5d1dcdc69,2016-09-19 23:11:33,44.99,2.83


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
37547,6916ca4502d6d3bfd39818759d55d536,bfbd0f9bdef84302105ad712db648a6c,1,NaN,nao recebi o produto e nem resposta da empresa,2016-10-06,2016-10-07 18:32:28


In [18]:
# ------------------------------------------------------------
# CALCULATE ORDER 30710 VALUE
# ------------------------------------------------------------
# We calculate the total product price and freight recorded
# for this order, since there is no payment record to compare
# against.
# ------------------------------------------------------------

v = i[["price", "freight_value"]].sum()

display(v.to_frame("value"))

,value
price,134.97
freight_value,8.49


In [19]:
# ------------------------------------------------------------
# ANALYZE REVIEWS CREATED BEFORE RECORDED DELIVERY
# ------------------------------------------------------------
# We already found 8,320 reviews where the review date is
# earlier than the recorded customer delivery date.
#
# Now we check:
# 1. What was the order status?
# 2. How many eventually have a delivery date?
# 3. How many reviews happened before shipment?
# 4. How many happened before approval/purchase?
# 5. What review scores were given?
# ------------------------------------------------------------

r = reviews.merge(
    orders[
        [
            "order_id",
            "order_status",
            "order_purchase_timestamp",
            "order_approved_at",
            "order_delivered_carrier_date",
            "order_delivered_customer_date"
        ]
    ],
    on="order_id",
    how="left"
)

# Find reviews created before the recorded delivery date
r = r[
    r["review_creation_date"] < r["order_delivered_customer_date"]
].copy()

# Create flags for each chronological inconsistency
r["before_purchase"] = (
    r["review_creation_date"] < r["order_purchase_timestamp"]
)

r["before_approved"] = (
    r["review_creation_date"] < r["order_approved_at"]
)

r["before_shipped"] = (
    r["review_creation_date"] < r["order_delivered_carrier_date"]
)

# Summarize the anomalous reviews
summary = pd.DataFrame({
    "total": [len(r)],
    "before_purchase": [r["before_purchase"].sum()],
    "before_approved": [r["before_approved"].sum()],
    "before_shipped": [r["before_shipped"].sum()],
    "score_1": [(r["review_score"] == 1).sum()],
    "score_2": [(r["review_score"] == 2).sum()],
    "score_3": [(r["review_score"] == 3).sum()],
    "score_4": [(r["review_score"] == 4).sum()],
    "score_5": [(r["review_score"] == 5).sum()]
})

display(summary)

,total,before_purchase,before_approved,before_shipped,score_1,score_2,score_3,score_4,score_5
0,8320,6,6,271,3531,569,766,999,2455


In [20]:
# ------------------------------------------------------------
# ANALYZE REVIEWS CREATED BEFORE RECORDED DELIVERY
# ------------------------------------------------------------
# We already found 8,320 reviews where the review date is
# earlier than the recorded customer delivery date.
#
# Now we check:
# 1. What was the order status?
# 2. How many eventually have a delivery date?
# 3. How many reviews happened before shipment?
# 4. How many happened before approval/purchase?
# 5. What review scores were given?
# ------------------------------------------------------------

r = reviews.merge(
    orders[
        [
            "order_id",
            "order_status",
            "order_purchase_timestamp",
            "order_approved_at",
            "order_delivered_carrier_date",
            "order_delivered_customer_date"
        ]
    ],
    on="order_id",
    how="left"
)

# Find reviews created before the recorded delivery date
r = r[
    r["review_creation_date"] < r["order_delivered_customer_date"]
].copy()

# Create flags for each chronological inconsistency
r["before_purchase"] = (
    r["review_creation_date"] < r["order_purchase_timestamp"]
)

r["before_approved"] = (
    r["review_creation_date"] < r["order_approved_at"]
)

r["before_shipped"] = (
    r["review_creation_date"] < r["order_delivered_carrier_date"]
)

# Summarize the anomalous reviews
summary = pd.DataFrame({
    "total": [len(r)],
    "before_purchase": [r["before_purchase"].sum()],
    "before_approved": [r["before_approved"].sum()],
    "before_shipped": [r["before_shipped"].sum()],
    "score_1": [(r["review_score"] == 1).sum()],
    "score_2": [(r["review_score"] == 2).sum()],
    "score_3": [(r["review_score"] == 3).sum()],
    "score_4": [(r["review_score"] == 4).sum()],
    "score_5": [(r["review_score"] == 5).sum()]
})

display(summary)

,total,before_purchase,before_approved,before_shipped,score_1,score_2,score_3,score_4,score_5
0,8320,6,6,271,3531,569,766,999,2455


In [21]:
# ------------------------------------------------------------
# ORDER STATUS OF REVIEWS MADE BEFORE RECORDED DELIVERY
# ------------------------------------------------------------

status = (
    r.groupby("order_status")
     .size()
     .reset_index(name="reviews")
     .sort_values("reviews", ascending=False)
)

display(status)

,order_status,reviews
1,delivered,8319
0,canceled,1


In [22]:
# ------------------------------------------------------------
# INSPECT REVIEWS CREATED BEFORE PURCHASE
# ------------------------------------------------------------
# These are the strongest chronological anomalies because
# a review appears before the order was even purchased.
# ------------------------------------------------------------

x = r[
    r["before_purchase"]
][
    [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "review_creation_date",
        "review_score"
    ]
].rename(columns={
    "order_id": "order",
    "order_status": "status",
    "order_purchase_timestamp": "purchase",
    "order_approved_at": "approved",
    "order_delivered_carrier_date": "shipped",
    "order_delivered_customer_date": "delivered",
    "review_creation_date": "review",
    "review_score": "score"
})

display(x)

,order,status,purchase,approved,shipped,delivered,review,score
24192,ebc94658c583ab37ad4f8e9091c4bef2,delivered,2018-04-14 08:22:43,2018-04-24 18:04:50,2018-04-26 15:57:00,2018-04-30 20:39:01,2018-03-30,2
24262,82fd1196a459f594fb1d66e667fc74c4,delivered,2018-04-10 21:09:18,2018-04-11 09:30:27,2018-04-17 16:04:51,2018-04-21 00:51:34,2018-01-23,4
41449,96f5be02bc9ffc589f3274500a64a7e2,delivered,2018-04-23 14:31:55,2018-04-27 16:11:21,2018-05-11 12:26:00,2018-05-16 18:19:00,2018-04-04,1
56320,4bb9c2002502ca416276dc1ff5efb1b3,delivered,2018-06-23 14:53:03,2018-06-27 14:20:08,2018-06-29 15:30:00,2018-07-17 17:08:47,2018-05-18,1
62675,4a62fb19d5fa08fb514619dfcc617b3d,delivered,2018-05-09 19:23:46,2018-05-10 14:32:42,2018-05-11 14:44:00,2018-05-15 18:12:35,2018-05-05,3
92017,450c49623c365a4edcf0c5a2c93aa7c9,delivered,2017-03-01 08:08:10,2017-03-02 15:32:28,2017-03-03 13:42:55,2017-03-08 10:26:04,2017-02-22,1


In [23]:
# ------------------------------------------------------------
# LOAD ORDER ITEMS AND PAYMENTS
# ------------------------------------------------------------
# We need these two tables to investigate the 381
# orders where item totals and payment totals differ.
# ------------------------------------------------------------

from pathlib import Path
import pandas as pd

DATA_DIR = Path("../data/raw")

order_items = pd.read_csv(
    DATA_DIR / "olist_order_items_dataset.csv"
)

payments = pd.read_csv(
    DATA_DIR / "olist_order_payments_dataset.csv"
)

print("Order items:", len(order_items))
print("Payments:", len(payments))

Order items: 112650
Payments: 103886


In [24]:
# ------------------------------------------------------------
# CALCULATE ORDER-LEVEL FINANCIAL DIFFERENCES
# ------------------------------------------------------------
# Item total = sum of product price + freight for each order
# Payment total = sum of all payment records for each order
#
# Then we compare the two totals.
# ------------------------------------------------------------

# Calculate total item value for each order
item_total = (
    order_items.groupby("order_id")
    .agg(
        price=("price", "sum"),
        freight=("freight_value", "sum")
    )
)

item_total["item_total"] = (
    item_total["price"] + item_total["freight"]
)

# Calculate total payment value for each order
pay_total = (
    payments.groupby("order_id")["payment_value"]
    .sum()
    .rename("payment_total")
)

# Combine the two totals
diff = item_total[["item_total"]].join(
    pay_total,
    how="inner"
)

# Calculate payment - item total
diff["difference"] = (
    diff["payment_total"] - diff["item_total"]
)

# Keep only meaningful differences (> $0.01)
d = diff[
    diff["difference"].abs() > 0.01
].copy()

print("Orders with differences:", len(d))

Orders with differences: 381


In [25]:
# ------------------------------------------------------------
# INVESTIGATE WHETHER MULTIPLE ITEMS/PAYMENTS EXPLAIN
# THE FINANCIAL DIFFERENCES
# ------------------------------------------------------------

# Count payment records per order
p = (
    payments.groupby("order_id")
    .size()
    .rename("payments")
)

# Count item records per order
i = (
    order_items.groupby("order_id")
    .size()
    .rename("items")
)

# Add payment and item counts
d = d.join(p, on="order_id")
d = d.join(i, on="order_id")

# Classify orders as single or multiple
d["payment_type"] = d["payments"].apply(
    lambda x: "multiple" if x > 1 else "single"
)

d["item_type"] = d["items"].apply(
    lambda x: "multiple" if x > 1 else "single"
)

# Count discrepancy orders in each combination
x = (
    d.groupby(["payment_type", "item_type"])
    .size()
    .reset_index(name="orders")
)

display(x)

,payment_type,item_type,orders
0,multiple,multiple,7
1,multiple,single,14
2,single,multiple,138
3,single,single,222


In [26]:
# ------------------------------------------------------------
# SUMMARIZE FINANCIAL DISCREPANCIES
# ------------------------------------------------------------
# We separate orders where payment is higher/lower than
# the calculated item + freight total.
# ------------------------------------------------------------

d["direction"] = d["difference"].apply(
    lambda x: "payment higher" if x > 0 else "payment lower"
)

x = (
    d.groupby("direction")["difference"]
    .agg(
        orders="count",
        min="min",
        max="max",
        mean="mean",
        median="median"
    )
    .reset_index()
)

display(x)

,direction,orders,min,max,mean,median
0,payment higher,291,0.01,182.81,10.551237,5.55
1,payment lower,90,-51.62,-0.01,-2.217667,-0.01


In [27]:
# ------------------------------------------------------------
# INSPECT THE LARGEST FINANCIAL DISCREPANCIES
# ------------------------------------------------------------
# These are the orders with the biggest absolute difference.
# We include item/payment counts to help identify a pattern.
# ------------------------------------------------------------

x = (
    d.assign(abs_diff=d["difference"].abs())
     .sort_values("abs_diff", ascending=False)
     .head(20)
     .reset_index()
)

x = x[
    [
        "order_id",
        "item_total",
        "payment_total",
        "difference",
        "abs_diff",
        "items",
        "payments"
    ]
].rename(columns={
    "order_id": "order",
    "item_total": "items_total",
    "payment_total": "payment_total",
    "difference": "difference",
    "abs_diff": "abs_difference",
    "items": "item_count",
    "payments": "payment_count"
})

display(x)

,order,items_total,payment_total,difference,abs_difference,item_count,payment_count
0,ce6d150fb29ada17d2082f4847107665,1403.66,1586.47,182.81,182.81,1,1
1,6e5fe7366a2e1bfbf3257dba0af1267f,287.91,406.92,119.01,119.01,6,1
2,70b742795bc441e94a44a084b6d9ce7a,466.93,578.82,111.89,111.89,1,1
3,996c7e73600ad3723e8627ab7bef81e4,587.90,664.43,76.53,76.53,1,1
4,70b7e94ea46d3e8b5bc12a50186edaf0,213.15,274.84,61.69,61.69,3,1
5,bc2c82b0ef78d2252b6176d1972db7c9,242.01,303.02,61.01,61.01,3,1
6,af9ffff2ce6b3defd34fd4c78857a379,413.17,466.97,53.80,53.80,1,1
7,262118ce178bb3e4590a3adcf6d62e6b,177.74,126.12,-51.62,51.62,2,1
8,bfdb5bbb06458d600a33d61f5f287472,348.93,394.36,45.43,45.43,1,1
9,8d9c0dc8d5a2ce804f6b925d8f8e6c1d,254.45,293.89,39.44,39.44,2,1


In [28]:
# ------------------------------------------------------------
# PAYMENT TYPES IN FINANCIAL DISCREPANCIES
# ------------------------------------------------------------
# We check whether certain payment methods are more common
# among the 381 orders with financial discrepancies.
# ------------------------------------------------------------

# Get payment types for each discrepant order
pt = (
    payments[payments["order_id"].isin(d.index)]
    .groupby("order_id")["payment_type"]
    .agg(lambda x: ", ".join(sorted(x.unique())))
    .rename("payment_method")
)

# Add payment method to discrepancy table
d2 = d.join(pt)

# Count discrepancies by payment method
x = (
    d2.groupby("payment_method")
    .size()
    .reset_index(name="orders")
    .sort_values("orders", ascending=False)
)

display(x)

,payment_method,orders
1,credit_card,335
0,boleto,26
2,"credit_card, voucher",10
3,debit_card,9
4,voucher,1


In [29]:
# ------------------------------------------------------------
# FINANCIAL DIFFERENCE BY PAYMENT METHOD
# ------------------------------------------------------------
# This tells us whether some payment methods have
# systematically larger discrepancies.
# ------------------------------------------------------------

x = (
    d2.groupby("payment_method")["difference"]
    .agg(
        orders="count",
        min="min",
        max="max",
        mean="mean",
        median="median"
    )
    .reset_index()
    .sort_values("orders", ascending=False)
)

display(x)

,payment_method,orders,min,max,mean,median
1,credit_card,335,-51.62,182.81,8.672328,4.230
0,boleto,26,-0.04,0.03,0.003077,0.010
2,"credit_card, voucher",10,-0.01,9.26,1.747000,0.965
3,debit_card,9,-16.50,0.01,-5.772222,-2.600
4,voucher,1,-0.01,-0.01,-0.010000,-0.010


In [30]:
# ------------------------------------------------------------
# COMPARE NORMAL ORDERS VS FINANCIALLY DISCREPANT ORDERS
# ------------------------------------------------------------
# We compare orders with:
#   difference <= $0.01  → normal
#   difference > $0.01   → discrepancy
#
# This helps us identify what characteristics are associated
# with the financial discrepancies.
# ------------------------------------------------------------

# Add item count and payment count for each order
i = order_items.groupby("order_id").size().rename("items")
p = payments.groupby("order_id").size().rename("payments")

# Add these counts to the discrepancy table
d = diff.copy()

d = d.join(i, how="left")
d = d.join(p, how="left")

# Replace missing counts with 0
d[["items", "payments"]] = d[["items", "payments"]].fillna(0)

# Classify orders as normal or discrepant
d["type"] = "normal"
d.loc[d["difference"].abs() > 0.01, "type"] = "discrepant"

# Create a summary table
summary = d.groupby("type").agg(
    orders=("difference", "size"),
    avg_diff=("difference", "mean"),
    median_diff=("difference", "median"),
    avg_items=("items", "mean"),
    avg_payments=("payments", "mean")
)

summary

,orders,avg_diff,median_diff,avg_items,avg_payments
type,,,,,
discrepant,381,7.534961,3.09,2.141732,1.060367
normal,98284,-0.000004,0.00,1.137835,1.044443


In [31]:
# ------------------------------------------------------------
# CHECK WHETHER FINANCIAL DISCREPANCIES INCREASE WITH
# THE NUMBER OF ITEMS IN AN ORDER
# ------------------------------------------------------------

# Count items in each order
ic = order_items.groupby("order_id").size().rename("items")

# Add item count to discrepant orders
x = diff[abs(diff["difference"]) > 0.01].copy()
x = x.join(ic)

# Group discrepancies by number of items
item_check = x.groupby("items").agg(
    orders=("difference", "size"),
    avg_diff=("difference", "mean"),
    median_diff=("difference", "median")
)

item_check

,orders,avg_diff,median_diff
items,,,
1,236,1.019271e+01,6.255
2,41,4.543415e+00,0.010
3,36,3.405556e+00,-0.010
4,26,1.441538e+00,-0.010
5,11,-1.291896e-15,-0.010
6,16,7.435000e+00,-0.015
7,8,6.250000e-03,0.015
8,3,-1.000000e-02,-0.020
11,2,2.000000e-02,0.020


In [32]:
# ------------------------------------------------------------
# CHECK PRICE AND FREIGHT FOR DISCREPANT ORDERS
# ------------------------------------------------------------
# We compare product price, freight, and payment difference
# to see whether freight is related to the discrepancy.
# ------------------------------------------------------------

x = (
    order_items.groupby("order_id")
    .agg(
        price=("price", "sum"),
        freight=("freight_value", "sum"),
        items=("order_item_id", "count")
    )
)

# Add financial difference
x = x.join(diff["difference"], how="inner")

# Keep only discrepant orders
x = x[abs(x["difference"]) > 0.01]

# Show the largest discrepancies first
x["abs_diff"] = x["difference"].abs()

x = (
    x.sort_values("abs_diff", ascending=False)
    .head(20)
    .reset_index()
)

display(x)

,order_id,price,freight,items,difference,abs_diff
0,ce6d150fb29ada17d2082f4847107665,1299.00,104.66,1,182.81,182.81
1,6e5fe7366a2e1bfbf3257dba0af1267f,179.19,108.72,6,119.01,119.01
2,70b742795bc441e94a44a084b6d9ce7a,269.99,196.94,1,111.89,111.89
3,996c7e73600ad3723e8627ab7bef81e4,559.90,28.00,1,76.53,76.53
4,70b7e94ea46d3e8b5bc12a50186edaf0,167.88,45.27,3,61.69,61.69
5,bc2c82b0ef78d2252b6176d1972db7c9,165.00,77.01,3,61.01,61.01
6,af9ffff2ce6b3defd34fd4c78857a379,395.65,17.52,1,53.80,53.80
7,262118ce178bb3e4590a3adcf6d62e6b,119.80,57.94,2,-51.62,51.62
8,bfdb5bbb06458d600a33d61f5f287472,297.00,51.93,1,45.43,45.43
9,8d9c0dc8d5a2ce804f6b925d8f8e6c1d,209.80,44.65,2,39.44,39.44


In [33]:
# ------------------------------------------------------------
# CHECK INSTALLMENTS IN FINANCIAL DISCREPANCIES
# ------------------------------------------------------------
# We check whether discrepant orders have unusual
# payment installment counts.
# ------------------------------------------------------------

p = (
    payments.groupby("order_id")
    .agg(
        installments=("payment_installments", "max"),
        payments=("payment_sequential", "count")
    )
)

# Add payment information to discrepant orders
x = diff[abs(diff["difference"]) > 0.01].copy()
x = x.join(p)

# Summarize discrepancies by installment count
inst = (
    x.groupby("installments")
    .agg(
        orders=("difference", "size"),
        avg_diff=("difference", "mean"),
        median_diff=("difference", "median")
    )
    .reset_index()
    .sort_values("orders", ascending=False)
)

display(inst)

,installments,orders,avg_diff,median_diff
0,1,63,-1.111905,-0.010
9,10,56,18.801964,13.605
2,3,50,1.722600,2.160
3,4,48,2.576250,3.210
5,6,38,6.886842,6.930
4,5,33,3.960606,5.480
1,2,23,0.596087,0.020
7,8,20,12.015000,10.950
6,7,18,10.048333,8.725
11,12,14,24.830714,21.630


In [34]:
# ------------------------------------------------------------
# CHECK HIGH-INSTALLMENT ORDERS
# ------------------------------------------------------------
# We inspect payment method, installments, payment value,
# and financial difference for orders with many installments.
# ------------------------------------------------------------

x = diff[abs(diff["difference"]) > 0.01].copy()

# Add payment information
p = payments[
    [
        "order_id",
        "payment_type",
        "payment_installments",
        "payment_value"
    ]
]

x = x.reset_index().merge(p, on="order_id", how="left")

# Keep orders with 9 or more installments
x = x[x["payment_installments"] >= 9]

# Show the largest discrepancies first
x = x.sort_values("difference", ascending=False)

display(x)

,order_id,item_total,payment_total,difference,payment_type,payment_installments,payment_value
330,ce6d150fb29ada17d2082f4847107665,1403.66,1586.47,182.81,credit_card,10,1586.47
177,6e5fe7366a2e1bfbf3257dba0af1267f,287.91,406.92,119.01,credit_card,10,406.92
184,70b742795bc441e94a44a084b6d9ce7a,466.93,578.82,111.89,credit_card,20,578.82
249,996c7e73600ad3723e8627ab7bef81e4,587.90,664.43,76.53,credit_card,10,664.43
185,70b7e94ea46d3e8b5bc12a50186edaf0,213.15,274.84,61.69,credit_card,24,274.84
...,...,...,...,...,...,...,...
182,6fd59e3ae7e24c50131f6bc97c4c7776,2913.48,2913.47,-0.01,credit_card,10,2913.47
162,674eb13687ba86abeaee12ece9ce0309,148.00,147.98,-0.02,credit_card,10,147.98
229,8fbcb92faf1aa60361f61ed7ae721a7e,140.03,140.01,-0.02,credit_card,10,140.01
351,df13cbaf3230c62a4b582317936e8a39,297.04,297.02,-0.02,credit_card,10,297.02


In [35]:
# ------------------------------------------------------------
# CHECK WHETHER LARGER ORDERS HAVE LARGER DISCREPANCIES
# ------------------------------------------------------------
# We compare the order's item total with the absolute
# financial difference.
# ------------------------------------------------------------

x = diff[abs(diff["difference"]) > 0.01].copy()

# Calculate absolute difference
x["abs_diff"] = x["difference"].abs()

# Create order-value ranges
x["value_range"] = pd.cut(
    x["item_total"],
    bins=[0, 100, 250, 500, 1000, 2000, float("inf")],
    labels=[
        "<100",
        "100-250",
        "250-500",
        "500-1000",
        "1000-2000",
        ">2000"
    ]
)

# Summarize discrepancies by order value
v = (
    x.groupby("value_range", observed=True)
    .agg(
        orders=("difference", "size"),
        avg_diff=("difference", "mean"),
        median_diff=("difference", "median"),
        max_diff=("abs_diff", "max")
    )
    .reset_index()
)

display(v)

,value_range,orders,avg_diff,median_diff,max_diff
0,<100,178,4.634494,4.03,19.59
1,100-250,125,9.261040,6.63,61.69
2,250-500,55,11.434000,0.01,119.01
3,500-1000,14,5.461429,-0.01,76.53
4,1000-2000,8,22.866250,0.02,182.81
5,>2000,1,-0.010000,-0.01,0.01


In [36]:
# ------------------------------------------------------------
# BUILD CUSTOMER-ORDER DATASET
# ------------------------------------------------------------
# We keep delivered orders because these represent completed
# purchases that actually reached the customer.
# ------------------------------------------------------------

# Keep only delivered orders
o = orders[orders["order_status"] == "delivered"].copy()

# Keep the columns needed for LTV analysis
o = o[
    [
        "order_id",
        "customer_id",
        "order_purchase_timestamp"
    ]
].rename(columns={
    "order_id": "order",
    "customer_id": "customer",
    "order_purchase_timestamp": "purchase"
})

# Display the resulting customer-order table
display(o.head())

,order,customer,purchase
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,2017-10-02 10:56:33
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,2018-07-24 20:41:37
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,2018-08-08 08:38:49
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,2017-11-18 19:28:06
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,2018-02-13 21:18:39


In [37]:
# ------------------------------------------------------------
# CALCULATE ORDER-LEVEL REVENUE
# ------------------------------------------------------------
# An order can contain multiple items.
# We therefore sum price + freight for each order.
# ------------------------------------------------------------

# Calculate total item value for each order
i = (
    order_items
    .groupby("order_id")
    .agg(
        price=("price", "sum"),
        freight=("freight_value", "sum")
    )
    .reset_index()
)

# Rename columns to keep them short
i = i.rename(columns={
    "order_id": "order"
})

# Calculate total order value
i["value"] = i["price"] + i["freight"]

# Display the order-level revenue table
display(i.head())

,order,price,freight,value
0,00010242fe8c5a6d1ba2dd792cb16214,58.90,13.29,72.19
1,00018f77f2f0320c557190d7a144bdd3,239.90,19.93,259.83
2,000229ec398224ef6ca0657da4fc703e,199.00,17.87,216.87
3,00024acbcdf0a6daa1e931b038114c75,12.99,12.79,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,199.90,18.14,218.04


In [38]:
# ------------------------------------------------------------
# BUILD CUSTOMER PURCHASE HISTORY
# ------------------------------------------------------------
# Merge the delivered orders with their order-level revenue.
# Each row now represents one completed customer order.
# ------------------------------------------------------------

# Merge order dates/customer info with order value
c = o.merge(
    i,
    on="order",
    how="left"
)

# Display the customer purchase history
display(c.head())

,order,customer,purchase,price,freight,value
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,2017-10-02 10:56:33,29.99,8.72,38.71
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,2018-07-24 20:41:37,118.70,22.76,141.46
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,2018-08-08 08:38:49,159.90,19.22,179.12
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,2017-11-18 19:28:06,45.00,27.20,72.20
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,2018-02-13 21:18:39,19.90,8.72,28.62


In [39]:
# ------------------------------------------------------------
# CREATE CUSTOMER-LEVEL LTV FEATURES
# ------------------------------------------------------------
# We convert the customer-order table into one row per customer.
# These are the basic features needed for LTV analysis.
# ------------------------------------------------------------

# Summarize each customer's purchase history
ltv = (
    c.groupby("customer")
    .agg(
        orders=("order", "count"),
        spend=("value", "sum"),
        avg_order=("value", "mean"),
        first_purchase=("purchase", "min"),
        last_purchase=("purchase", "max")
    )
    .reset_index()
)

# Display the customer-level table
display(ltv.head())

,customer,orders,spend,avg_order,first_purchase,last_purchase
0,00012a2ce6f8dcda20d059ce98491703,1,114.74,114.74,2017-11-14 16:08:26,2017-11-14 16:08:26
1,000161a058600d5901f007fab4c27140,1,67.41,67.41,2017-07-16 09:40:32,2017-07-16 09:40:32
2,0001fd6190edaaf884bcaf3d49edf079,1,195.42,195.42,2017-02-28 11:06:43,2017-02-28 11:06:43
3,0002414f95344307404f0ace7a26f1d5,1,179.35,179.35,2017-08-16 13:09:20,2017-08-16 13:09:20
4,000379cdec625522490c315e70c7a9fb,1,107.01,107.01,2018-04-02 13:42:17,2018-04-02 13:42:17


In [41]:
# ------------------------------------------------------------
# CHECK CUSTOMER ID STRUCTURE
# ------------------------------------------------------------
# Olist has:
#   customer_id        -> linked to a specific order
#   customer_unique_id -> identifies the actual customer
#
# We check whether repeat purchases exist when using
# customer_unique_id instead.
# ------------------------------------------------------------

# Add the real customer identifier to delivered orders
o = orders[orders["order_status"] == "delivered"][
    [
        "order_id",
        "customer_id",
        "order_purchase_timestamp"
    ]
].merge(
    customers[
        [
            "customer_id",
            "customer_unique_id"
        ]
    ],
    on="customer_id",
    how="left"
)

# Rename columns to keep them short
o = o.rename(columns={
    "order_id": "order",
    "customer_unique_id": "customer",
    "order_purchase_timestamp": "purchase"
})

# Count orders per actual customer
freq = (
    o.groupby("customer")
    .size()
    .value_counts()
    .sort_index()
    .rename_axis("orders")
    .reset_index(name="customers")
)

# Display the repeat-purchase distribution
display(freq)

,orders,customers
0,1,90557
1,2,2573
2,3,181
3,4,28
4,5,9
5,6,5
6,7,3
7,9,1
8,15,1


In [42]:
# ------------------------------------------------------------
# COMPARE ONE-TIME VS REPEAT CUSTOMERS
# ------------------------------------------------------------
# customer_id is tied to an order.
# customer_unique_id identifies the actual customer across
# multiple orders, so we use customer_unique_id for LTV.
# ------------------------------------------------------------

# Keep delivered orders and connect them to the real customer
o = orders[orders["order_status"] == "delivered"][
    ["order_id", "customer_id"]
].merge(
    customers[["customer_id", "customer_unique_id"]],
    on="customer_id",
    how="left"
)

# Calculate total payment for each order
p = payments.groupby("order_id")["payment_value"].sum()

# Attach order payment to each delivered order
o["spend"] = o["order_id"].map(p)

# Create one row per actual customer
c = o.groupby("customer_unique_id").agg(
    orders=("order_id", "nunique"),
    spend=("spend", "sum")
)

# Classify customers
c["type"] = c["orders"].apply(
    lambda x: "one-time" if x == 1 else "repeat"
)

# Compare one-time and repeat customers
result = c.groupby("type").agg(
    customers=("orders", "size"),
    avg_orders=("orders", "mean"),
    avg_spend=("spend", "mean"),
    median_spend=("spend", "median")
)

# Display as a Jupyter table
display(result)

,customers,avg_orders,avg_spend,median_spend
type,,,,
one-time,90557,1.000000,160.761781,105.39
repeat,2801,2.113888,308.588793,225.55


In [43]:
# ------------------------------------------------------------
# COMPARE ORDER VALUE: ONE-TIME VS REPEAT CUSTOMERS
# ------------------------------------------------------------
# We calculate the average order value for each customer
# and compare one-time customers with repeat customers.
# ------------------------------------------------------------

# Calculate average order value for each customer
a = (
    o.groupby("customer_unique_id")
    .agg(
        orders=("order_id", "nunique"),
        avg_order=("spend", "mean")
    )
)

# Classify customers
a["type"] = a["orders"].apply(
    lambda x: "one-time" if x == 1 else "repeat"
)

# Compare average order value
result = (
    a.groupby("type")
    .agg(
        customers=("orders", "size"),
        avg_order=("avg_order", "mean"),
        median_order=("avg_order", "median")
    )
)

# Display as a Jupyter table
display(result)

,customers,avg_order,median_order
type,,,
one-time,90557,160.763556,105.390
repeat,2801,145.868157,109.965


In [44]:
# ------------------------------------------------------------
# COMPARE TOTAL SPEND: ONE-TIME VS REPEAT CUSTOMERS
# ------------------------------------------------------------
# c already contains total spending for each actual customer.
# We compare the total lifetime spend of one-time and
# repeat customers.
# ------------------------------------------------------------

r = c.groupby("type").agg(
    customers=("spend", "size"),
    avg_spend=("spend", "mean"),
    median_spend=("spend", "median")
)

# Display as a Jupyter table
display(r)

,customers,avg_spend,median_spend
type,,,
one-time,90557,160.761781,105.39
repeat,2801,308.588793,225.55


In [45]:
# ------------------------------------------------------------
# TIME BETWEEN ORDERS FOR REPEAT CUSTOMERS
# ------------------------------------------------------------
# customer_unique_id identifies the actual customer.
# We calculate the number of days between their purchases.
# ------------------------------------------------------------

# Connect each order to the actual customer
o = orders[
    ["order_id", "customer_id", "order_purchase_timestamp"]
].merge(
    customers[
        ["customer_id", "customer_unique_id"]
    ],
    on="customer_id",
    how="left"
)

# Convert purchase time to datetime
o["order_purchase_timestamp"] = pd.to_datetime(
    o["order_purchase_timestamp"]
)

# Sort each customer's orders chronologically
o = o.sort_values(
    ["customer_unique_id", "order_purchase_timestamp"]
)

# Calculate the gap from the previous order
o["gap"] = (
    o.groupby("customer_unique_id")["order_purchase_timestamp"]
    .diff()
    .dt.total_seconds()
    / (60 * 60 * 24)
)

# Keep only repeat purchases
g = o[o["gap"].notna()]

# Summarize the time between orders
r = pd.DataFrame({
    "customers": [g["customer_unique_id"].nunique()],
    "avg_days": [g["gap"].mean()],
    "median_days": [g["gap"].median()],
    "min_days": [g["gap"].min()],
    "max_days": [g["gap"].max()]
})

# Display as a Jupyter table
display(r)

,customers,avg_days,median_days,min_days,max_days
0,2997,78.227702,28.330833,0.0,608.978912


In [46]:
# ------------------------------------------------------------
# REPEAT PURCHASE WINDOW
# ------------------------------------------------------------
# We group the time between orders into practical time ranges.
# This helps identify when customers commonly make another
# purchase.
# ------------------------------------------------------------

# Create time ranges for the gap between purchases
g["window"] = pd.cut(
    g["gap"],
    bins=[0, 7, 14, 30, 60, 90, 180, 365, float("inf")],
    labels=[
        "0-7 days",
        "8-14 days",
        "15-30 days",
        "31-60 days",
        "61-90 days",
        "91-180 days",
        "181-365 days",
        "365+ days"
    ]
)

# Count repeat purchases in each window
r = (
    g["window"]
    .value_counts()
    .sort_index()
    .rename_axis("window")
    .reset_index(name="orders")
)

# Display as a Jupyter table
display(r)

,window,orders
0,0-7 days,907
1,8-14 days,200
2,15-30 days,302
3,31-60 days,389
4,61-90 days,234
5,91-180 days,474
6,181-365 days,457
7,365+ days,90


In [47]:
# ------------------------------------------------------------
# CHECK SAME-DAY REPEAT ORDERS
# ------------------------------------------------------------
# A gap of 0 days may mean the customer placed multiple orders
# on the same day rather than returning immediately.
# ------------------------------------------------------------

# Find repeat orders with a gap of exactly 0 days
same_day = g[g["gap"] == 0].copy()

# Count customers and orders involved
r = pd.DataFrame({
    "customers": [same_day["customer_unique_id"].nunique()],
    "orders": [len(same_day)],
    "avg_orders_customer": [
        same_day.groupby("customer_unique_id")["order_id"].count().mean()
    ]
})

# Display as a Jupyter table
display(r)

,customers,orders,avg_orders_customer
0,290,292,1.006897


In [48]:
# ------------------------------------------------------------
# GENUINE REPEAT-PURCHASE WINDOW
# ------------------------------------------------------------
# Remove same-day orders (gap = 0).
# We only consider customers who returned on a later day.
# ------------------------------------------------------------

# Keep only repeat purchases made on a later day
g2 = g[g["gap"] > 0].copy()

# Create practical time windows
g2["window"] = pd.cut(
    g2["gap"],
    bins=[0, 7, 14, 30, 60, 90, 180, 365, float("inf")],
    labels=[
        "1-7 days",
        "8-14 days",
        "15-30 days",
        "31-60 days",
        "61-90 days",
        "91-180 days",
        "181-365 days",
        "365+ days"
    ]
)

# Count genuine repeat purchases in each window
r = (
    g2["window"]
    .value_counts()
    .sort_index()
    .rename_axis("window")
    .reset_index(name="orders")
)

# Display as a Jupyter table
display(r)

,window,orders
0,1-7 days,907
1,8-14 days,200
2,15-30 days,302
3,31-60 days,389
4,61-90 days,234
5,91-180 days,474
6,181-365 days,457
7,365+ days,90


In [49]:
# ============================================================
# AUDIT: TIME TO SECOND PURCHASE
# ============================================================
# Question:
# How long does a customer typically take to make their
# second purchase?
#
# This is important for deciding when a checkout incentive
# should be triggered.
# ============================================================

import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data/raw")

# Load required files
orders = pd.read_csv(
    DATA_DIR / "olist_orders_dataset.csv",
    parse_dates=[
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
)

customers = pd.read_csv(
    DATA_DIR / "olist_customers_dataset.csv"
)

# ------------------------------------------------------------
# 1. Keep only orders linked to customers
# ------------------------------------------------------------

df = orders.merge(
    customers[["customer_id", "customer_unique_id"]],
    on="customer_id",
    how="left"
)

# Only orders with a purchase timestamp
df = df.dropna(subset=["order_purchase_timestamp"])

# Sort purchases chronologically for each unique customer
df = df.sort_values(
    ["customer_unique_id", "order_purchase_timestamp"]
)

# ------------------------------------------------------------
# 2. Get first and second purchase dates
# ------------------------------------------------------------

df["purchase_number"] = (
    df.groupby("customer_unique_id")
      .cumcount() + 1
)

first_purchase = (
    df[df["purchase_number"] == 1]
    [["customer_unique_id", "order_purchase_timestamp"]]
    .rename(columns={
        "order_purchase_timestamp": "first_purchase"
    })
)

second_purchase = (
    df[df["purchase_number"] == 2]
    [["customer_unique_id", "order_purchase_timestamp"]]
    .rename(columns={
        "order_purchase_timestamp": "second_purchase"
    })
)

# ------------------------------------------------------------
# 3. Calculate days until second purchase
# ------------------------------------------------------------

repeat = first_purchase.merge(
    second_purchase,
    on="customer_unique_id",
    how="inner"
)

repeat["days_to_second_purchase"] = (
    repeat["second_purchase"] -
    repeat["first_purchase"]
).dt.total_seconds() / (60 * 60 * 24)

# ------------------------------------------------------------
# 4. Summary statistics
# ------------------------------------------------------------

summary = pd.DataFrame({
    "customers_with_repeat_purchase": [len(repeat)],
    "avg_days_to_second_purchase": [
        repeat["days_to_second_purchase"].mean()
    ],
    "median_days_to_second_purchase": [
        repeat["days_to_second_purchase"].median()
    ],
    "min_days": [
        repeat["days_to_second_purchase"].min()
    ],
    "max_days": [
        repeat["days_to_second_purchase"].max()
    ]
})

summary

,customers_with_repeat_purchase,avg_days_to_second_purchase,median_days_to_second_purchase,min_days,max_days
0,2997,80.349981,27.923773,0.0,608.978912


In [50]:
# ============================================================
# DISTRIBUTION OF TIME TO SECOND PURCHASE
# ============================================================

repeat["window"] = pd.cut(
    repeat["days_to_second_purchase"],
    bins=[0, 7, 14, 30, 60, 90, 180, 365, float("inf")],
    labels=[
        "1-7 days",
        "8-14 days",
        "15-30 days",
        "31-60 days",
        "61-90 days",
        "91-180 days",
        "181-365 days",
        "365+ days"
    ],
    include_lowest=True
)

window_analysis = (
    repeat.groupby("window", observed=False)
    .size()
    .reset_index(name="customers")
)

window_analysis["percentage"] = (
    window_analysis["customers"]
    / window_analysis["customers"].sum()
    * 100
)

window_analysis

,window,customers,percentage
0,1-7 days,1097,36.603270
1,8-14 days,166,5.538872
2,15-30 days,267,8.908909
3,31-60 days,320,10.677344
4,61-90 days,208,6.940274
5,91-180 days,424,14.147481
6,181-365 days,427,14.247581
7,365+ days,88,2.936270


In [51]:
# ============================================================
# TIME TO SECOND PURCHASE — CORRECT VERSION
# ============================================================

# Connect orders to the actual customer
p = orders[
    ["order_id", "customer_id", "order_purchase_timestamp"]
].merge(
    customers[
        ["customer_id", "customer_unique_id"]
    ],
    on="customer_id",
    how="left"
)

# Convert purchase timestamp
p["order_purchase_timestamp"] = pd.to_datetime(
    p["order_purchase_timestamp"]
)

# Sort by actual customer and purchase time
p = p.sort_values(
    ["customer_unique_id", "order_purchase_timestamp"]
)

# Get the first two purchases of each customer
p["n"] = (
    p.groupby("customer_unique_id")
     .cumcount() + 1
)

first = p[p["n"] == 1][
    ["customer_unique_id", "order_purchase_timestamp"]
].rename(
    columns={"order_purchase_timestamp": "first"}
)

second = p[p["n"] == 2][
    ["customer_unique_id", "order_purchase_timestamp"]
].rename(
    columns={"order_purchase_timestamp": "second"}
)

# Match first and second purchases
r = first.merge(
    second,
    on="customer_unique_id",
    how="inner"
)

# Calculate days to second purchase
r["days"] = (
    r["second"] - r["first"]
).dt.total_seconds() / (24 * 60 * 60)

# Summary
summary = pd.DataFrame({
    "customers": [len(r)],
    "avg_days": [r["days"].mean()],
    "median_days": [r["days"].median()],
    "min_days": [r["days"].min()],
    "max_days": [r["days"].max()]
})

display(summary)

,customers,avg_days,median_days,min_days,max_days
0,2997,80.349981,27.923773,0.0,608.978912


In [52]:
# ============================================================
# GENUINE TIME TO SECOND PURCHASE
# ============================================================
# Exclude customers whose first and second orders were placed
# on the same day.
# ============================================================

r2 = r[r["days"] > 0].copy()

summary = pd.DataFrame({
    "customers": [r2["customer_unique_id"].nunique()],
    "avg_days": [r2["days"].mean()],
    "median_days": [r2["days"].median()],
    "min_days": [r2["days"].min()],
    "max_days": [r2["days"].max()]
})

display(summary)

,customers,avg_days,median_days,min_days,max_days
0,2721,88.500144,39.096586,0.000012,608.978912


In [53]:
# ============================================================
# GENUINE REPEAT PURCHASES — 1+ DAY GAP
# ============================================================

r3 = r[r["days"] >= 1].copy()

summary = pd.DataFrame({
    "customers": [r3["customer_unique_id"].nunique()],
    "avg_days": [r3["days"].mean()],
    "median_days": [r3["days"].median()],
    "min_days": [r3["days"].min()],
    "max_days": [r3["days"].max()]
})

display(summary)

,customers,avg_days,median_days,min_days,max_days
0,2070,116.317494,74.828831,1.006968,608.978912


In [54]:
# ============================================================
# GENUINE REPEAT-PURCHASE WINDOWS
# ============================================================

bins = [1, 7, 14, 30, 60, 90, 180, 365, float("inf")]

labels = [
    "1-7 days",
    "8-14 days",
    "15-30 days",
    "31-60 days",
    "61-90 days",
    "91-180 days",
    "181-365 days",
    "365+ days"
]

r3["window"] = pd.cut(
    r3["days"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

w = (
    r3["window"]
    .value_counts()
    .reindex(labels, fill_value=0)
    .reset_index()
)

w.columns = ["window", "customers"]

w["percentage"] = (
    w["customers"] / len(r3) * 100
)

display(w)

,window,customers,percentage
0,1-7 days,170,8.212560
1,8-14 days,166,8.019324
2,15-30 days,267,12.898551
3,31-60 days,320,15.458937
4,61-90 days,208,10.048309
5,91-180 days,424,20.483092
6,181-365 days,427,20.628019
7,365+ days,88,4.251208


In [55]:
# ============================================================
# 02_LTV_MODELING
# CUSTOMER-LEVEL LTV DATASET
# ============================================================

# Connect orders to the real customer identifier
ltv_orders = orders[
    [
        "order_id",
        "customer_id",
        "order_purchase_timestamp",
        "order_status"
    ]
].merge(
    customers[
        [
            "customer_id",
            "customer_unique_id"
        ]
    ],
    on="customer_id",
    how="left"
)

# Convert timestamp
ltv_orders["order_purchase_timestamp"] = pd.to_datetime(
    ltv_orders["order_purchase_timestamp"]
)

# Keep only orders with a valid customer
ltv_orders = ltv_orders.dropna(
    subset=["customer_unique_id"]
)

# Payment value per order
ltv_payments = (
    payments
    .groupby("order_id")["payment_value"]
    .sum()
    .rename("order_value")
    .reset_index()
)

# Add payment value to orders
ltv_orders = ltv_orders.merge(
    ltv_payments,
    on="order_id",
    how="left"
)

# Missing payment = 0
ltv_orders["order_value"] = (
    ltv_orders["order_value"]
    .fillna(0)
)

# Customer-level LTV
customer_ltv = (
    ltv_orders
    .groupby("customer_unique_id")
    .agg(
        orders=("order_id", "nunique"),
        total_spend=("order_value", "sum"),
        first_purchase=("order_purchase_timestamp", "min"),
        last_purchase=("order_purchase_timestamp", "max")
    )
    .reset_index()
)

# Customer lifetime in days
customer_ltv["lifetime_days"] = (
    customer_ltv["last_purchase"]
    - customer_ltv["first_purchase"]
).dt.total_seconds() / (24 * 60 * 60)

display(customer_ltv.head())
print("Customers:", len(customer_ltv))

,customer_unique_id,orders,total_spend,first_purchase,last_purchase,lifetime_days
0,0000366f3b9a7992bf8c76cfdf3221e2,1,141.90,2018-05-10 10:56:27,2018-05-10 10:56:27,0.0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,27.19,2018-05-07 11:11:27,2018-05-07 11:11:27,0.0
2,0000f46a3911fa3c0805444483337064,1,86.22,2017-03-10 21:05:03,2017-03-10 21:05:03,0.0
3,0000f6ccb0745a6a4b88665a16c9f078,1,43.62,2017-10-12 20:29:41,2017-10-12 20:29:41,0.0
4,0004aac84e0df4da2b147fca70cf8255,1,196.89,2017-11-14 19:45:42,2017-11-14 19:45:42,0.0


Customers: 96096


In [56]:
# ============================================================
# BASIC LTV STATISTICS
# ============================================================

ltv_summary = pd.DataFrame({
    "customers": [len(customer_ltv)],
    "total_revenue": [customer_ltv["total_spend"].sum()],
    "avg_ltv": [customer_ltv["total_spend"].mean()],
    "median_ltv": [customer_ltv["total_spend"].median()],
    "min_ltv": [customer_ltv["total_spend"].min()],
    "max_ltv": [customer_ltv["total_spend"].max()]
})

display(ltv_summary)

,customers,total_revenue,avg_ltv,median_ltv,min_ltv,max_ltv
0,96096,16008872.12,166.592492,108.0,0.0,13664.08


In [57]:
# ============================================================
# ONE-TIME VS REPEAT CUSTOMER LTV
# ============================================================

customer_ltv["customer_type"] = np.where(
    customer_ltv["orders"] == 1,
    "one-time",
    "repeat"
)

ltv_by_type = (
    customer_ltv
    .groupby("customer_type")
    .agg(
        customers=("customer_unique_id", "count"),
        avg_orders=("orders", "mean"),
        avg_ltv=("total_spend", "mean"),
        median_ltv=("total_spend", "median")
    )
    .reset_index()
)

display(ltv_by_type)

,customer_type,customers,avg_orders,avg_ltv,median_ltv
0,one-time,93099,1.000000,161.815373,105.70
1,repeat,2997,2.116116,314.989226,225.84


In [58]:
# ============================================================
# REVENUE CONTRIBUTION
# ============================================================

revenue_by_type = (
    customer_ltv
    .groupby("customer_type")
    .agg(
        customers=("customer_unique_id", "count"),
        total_revenue=("total_spend", "sum")
    )
    .reset_index()
)

revenue_by_type["customer_percentage"] = (
    revenue_by_type["customers"]
    / revenue_by_type["customers"].sum()
    * 100
)

revenue_by_type["revenue_percentage"] = (
    revenue_by_type["total_revenue"]
    / revenue_by_type["total_revenue"].sum()
    * 100
)

display(revenue_by_type)

,customer_type,customers,total_revenue,customer_percentage,revenue_percentage
0,one-time,93099,15064849.41,96.881244,94.103128
1,repeat,2997,944022.71,3.118756,5.896872


In [59]:
# ============================================================
# LTV VALUE BANDS
# ============================================================

bins = [
    0,
    50,
    100,
    250,
    500,
    1000,
    2000,
    float("inf")
]

labels = [
    "<50",
    "50-100",
    "100-250",
    "250-500",
    "500-1000",
    "1000-2000",
    "2000+"
]

customer_ltv["ltv_band"] = pd.cut(
    customer_ltv["total_spend"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

ltv_distribution = (
    customer_ltv["ltv_band"]
    .value_counts()
    .reindex(labels, fill_value=0)
    .reset_index()
)

ltv_distribution.columns = [
    "ltv_band",
    "customers"
]

ltv_distribution["percentage"] = (
    ltv_distribution["customers"]
    / len(customer_ltv)
    * 100
)

display(ltv_distribution)

,ltv_band,customers,percentage
0,<50,15849,16.492882
1,50-100,28546,29.705711
2,100-250,37454,38.975608
3,250-500,9758,10.154429
4,500-1000,3271,3.403888
5,1000-2000,993,1.033342
6,2000+,225,0.234141


In [60]:
# ============================================================
# REVENUE CONCENTRATION
# ============================================================

ltv_sorted = customer_ltv.sort_values(
    "total_spend",
    ascending=False
).reset_index(drop=True)

ltv_sorted["cumulative_revenue"] = (
    ltv_sorted["total_spend"].cumsum()
)

total_revenue = ltv_sorted["total_spend"].sum()

ltv_sorted["cumulative_revenue_pct"] = (
    ltv_sorted["cumulative_revenue"]
    / total_revenue
    * 100
)

# Revenue generated by top customer groups
concentration = pd.DataFrame({
    "segment": [
        "Top 1%",
        "Top 5%",
        "Top 10%",
        "Top 20%"
    ],
    "revenue_percentage": [
        ltv_sorted.head(max(1, int(len(ltv_sorted) * 0.01)))["total_spend"].sum() / total_revenue * 100,
        ltv_sorted.head(max(1, int(len(ltv_sorted) * 0.05)))["total_spend"].sum() / total_revenue * 100,
        ltv_sorted.head(max(1, int(len(ltv_sorted) * 0.10)))["total_spend"].sum() / total_revenue * 100,
        ltv_sorted.head(max(1, int(len(ltv_sorted) * 0.20)))["total_spend"].sum() / total_revenue * 100
    ]
})

display(concentration)

,segment,revenue_percentage
0,Top 1%,10.476648
1,Top 5%,27.035498
2,Top 10%,38.511550
3,Top 20%,53.769677


In [61]:
# ============================================================
# LTV BY ORDER FREQUENCY
# ============================================================

ltv_by_orders = (
    customer_ltv
    .groupby("orders")
    .agg(
        customers=("customer_unique_id", "count"),
        avg_ltv=("total_spend", "mean"),
        median_ltv=("total_spend", "median"),
        total_revenue=("total_spend", "sum")
    )
    .reset_index()
    .sort_values("orders")
)

display(ltv_by_orders)

,orders,customers,avg_ltv,median_ltv,total_revenue
0,1,93099,161.815373,105.700,15064849.41
1,2,2745,294.847137,217.740,809355.39
2,3,203,473.388325,342.480,96097.83
3,4,30,779.129333,537.720,23373.88
4,5,8,759.656250,674.655,6077.25
5,6,6,696.251667,743.630,4177.51
6,7,3,946.853333,959.010,2840.56
7,9,1,1172.660000,1172.660,1172.66
8,17,1,927.630000,927.630,927.63


In [62]:
# ============================================================
# HIGH-VALUE CUSTOMERS
# ============================================================

ltv_threshold = customer_ltv["total_spend"].quantile(0.90)

customer_ltv["high_value"] = np.where(
    customer_ltv["total_spend"] >= ltv_threshold,
    "high-value",
    "standard"
)

high_value_summary = (
    customer_ltv
    .groupby("high_value")
    .agg(
        customers=("customer_unique_id", "count"),
        avg_ltv=("total_spend", "mean"),
        median_ltv=("total_spend", "median"),
        total_revenue=("total_spend", "sum")
    )
    .reset_index()
)

high_value_summary["revenue_percentage"] = (
    high_value_summary["total_revenue"]
    / customer_ltv["total_spend"].sum()
    * 100
)

print("High-value threshold:", ltv_threshold)

display(high_value_summary)

High-value threshold: 319.57


,high_value,customers,avg_ltv,median_ltv,total_revenue,revenue_percentage
0,high-value,9611,641.546554,476.14,6165903.93,38.515542
1,standard,86485,113.811276,97.77,9842968.19,61.484458


In [63]:
# ============================================================
# LTV + REPEAT BEHAVIOR
# ============================================================

behavior_ltv = (
    customer_ltv
    .groupby("customer_type")
    .agg(
        customers=("customer_unique_id", "count"),
        avg_orders=("orders", "mean"),
        avg_ltv=("total_spend", "mean"),
        median_ltv=("total_spend", "median"),
        avg_lifetime_days=("lifetime_days", "mean"),
        median_lifetime_days=("lifetime_days", "median")
    )
    .reset_index()
)

display(behavior_ltv)

,customer_type,customers,avg_orders,avg_ltv,median_ltv,avg_lifetime_days,median_lifetime_days
0,one-time,93099,1.000000,161.815373,105.70,0.000000,0.000000
1,repeat,2997,2.116116,314.989226,225.84,87.311199,33.662755


In [64]:
# ============================================================
# CUSTOMER VALUE + BEHAVIOR SEGMENTS
# ============================================================

ltv_cutoff = customer_ltv["total_spend"].median()

customer_ltv["value_segment"] = np.where(
    customer_ltv["total_spend"] >= ltv_cutoff,
    "high-value",
    "low-value"
)

customer_ltv["incentive_segment"] = np.select(
    [
        (
            (customer_ltv["value_segment"] == "high-value") &
            (customer_ltv["customer_type"] == "repeat")
        ),
        (
            (customer_ltv["value_segment"] == "high-value") &
            (customer_ltv["customer_type"] == "one-time")
        ),
        (
            (customer_ltv["value_segment"] == "low-value") &
            (customer_ltv["customer_type"] == "repeat")
        )
    ],
    [
        "high-value repeat",
        "high-value one-time",
        "low-value repeat"
    ],
    default="low-value one-time"
)

segment_summary = (
    customer_ltv
    .groupby("incentive_segment")
    .agg(
        customers=("customer_unique_id", "count"),
        avg_orders=("orders", "mean"),
        avg_ltv=("total_spend", "mean"),
        median_ltv=("total_spend", "median"),
        total_revenue=("total_spend", "sum")
    )
    .reset_index()
)

segment_summary["customer_percentage"] = (
    segment_summary["customers"]
    / len(customer_ltv)
    * 100
)

segment_summary["revenue_percentage"] = (
    segment_summary["total_revenue"]
    / customer_ltv["total_spend"].sum()
    * 100
)

display(segment_summary)

,incentive_segment,customers,avg_orders,avg_ltv,median_ltv,total_revenue,customer_percentage,revenue_percentage
0,high-value one-time,45432,1.000000,264.913421,180.210,12035546.53,47.277722,75.180478
1,high-value repeat,2634,2.129081,347.052285,251.975,914135.72,2.741009,5.710182
2,low-value one-time,47667,1.000000,63.551364,63.000,3029302.88,49.603521,18.922650
3,low-value repeat,363,2.022039,82.333306,85.070,29886.99,0.377747,0.186690


In [65]:
# ============================================================
# FINAL LTV MODEL OUTPUT
# ============================================================

final_ltv = customer_ltv[
    [
        "customer_unique_id",
        "orders",
        "total_spend",
        "first_purchase",
        "last_purchase",
        "lifetime_days",
        "customer_type",
        "high_value",
        "incentive_segment"
    ]
].copy()

final_ltv = final_ltv.sort_values(
    "total_spend",
    ascending=False
)

display(final_ltv.head(20))

,customer_unique_id,orders,total_spend,first_purchase,last_purchase,lifetime_days,customer_type,high_value,incentive_segment
3826,0a0a92112bd4c708ca5fde585afaa872,1,13664.08,2017-09-29 15:24:52,2017-09-29 15:24:52,0.000000,one-time,high-value,high-value one-time
26456,46450c74a0d8c5ca9395da1daac6c120,3,9553.02,2018-07-24 20:41:01,2018-08-17 20:06:36,23.976100,repeat,high-value,high-value repeat
81962,da122df9eeddfedc1dc1f5349a1a690c,2,7571.63,2017-04-01 15:58:40,2017-04-01 15:58:41,0.000012,repeat,high-value,high-value repeat
44447,763c8b1c9c68a0229c42c9fc6f662b93,1,7274.88,2018-07-15 14:49:44,2018-07-15 14:49:44,0.000000,one-time,high-value,high-value one-time
82808,dc4802a71eae9be1dd28f5d788ceb526,1,6929.31,2017-02-12 20:37:36,2017-02-12 20:37:36,0.000000,one-time,high-value,high-value one-time
26205,459bef486812aa25204be022145caa62,1,6922.21,2018-07-25 18:10:17,2018-07-25 18:10:17,0.000000,one-time,high-value,high-value one-time
95806,ff4159b92c40ebe40454e3e6a7c35ed6,1,6726.66,2017-05-24 18:14:34,2017-05-24 18:14:34,0.000000,one-time,high-value,high-value one-time
24121,4007669dec559734d6f53e029e360987,1,6081.54,2017-11-24 11:03:35,2017-11-24 11:03:35,0.000000,one-time,high-value,high-value one-time
35070,5d0a2980b292d049061542014e8960bf,1,4809.44,2018-07-12 12:08:36,2018-07-12 12:08:36,0.000000,one-time,high-value,high-value one-time
89688,eebb5dda148d3893cdaf5b5ca3040ccb,1,4764.34,2017-04-18 18:50:13,2017-04-18 18:50:13,0.000000,one-time,high-value,high-value one-time


In [86]:
# ------------------------------------------------------------
# CUSTOMER-LEVEL BASE DATASET
# ------------------------------------------------------------
# We use customer_unique_id as the actual customer identifier.
# Each row in the resulting dataframe represents one customer.
# ------------------------------------------------------------

# Get customer_unique_id for each order
customer_orders = (
    orders[
        [
            "order_id",
            "customer_id",
            "order_status",
            "order_purchase_timestamp",
            "order_delivered_customer_date"
        ]
    ]
    .merge(
        customers[
            [
                "customer_id",
                "customer_unique_id"
            ]
        ],
        on="customer_id",
        how="left"
    )
)

# Keep delivered orders for customer-value analysis
customer_orders = customer_orders[
    customer_orders["order_status"] == "delivered"
].copy()

customer_orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_delivered_customer_date,customer_unique_id
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-10 21:25:13,7c396fd4830fd04220f754e42b4e5bff
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-08-07 15:27:45,af07308b275d755c9edb36a90c618231
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-17 18:06:29,3a653a41f6f9fc3d2a113cf8398680e8
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-12-02 00:28:42,7c142cf63193a1473d2e66489a9ae977
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-16 18:17:02,72632f0f9dd73dfee390c9b22eb56dd6


In [87]:
# ------------------------------------------------------------
# CUSTOMER PURCHASE BEHAVIOUR
# ------------------------------------------------------------

customer_summary = (
    customer_orders
    .groupby("customer_unique_id")
    .agg(
        orders=("order_id", "nunique"),
        first_purchase=("order_purchase_timestamp", "min"),
        last_purchase=("order_purchase_timestamp", "max")
    )
    .reset_index()
)

customer_summary.head()

,customer_unique_id,orders,first_purchase,last_purchase
0,0000366f3b9a7992bf8c76cfdf3221e2,1,2018-05-10 10:56:27,2018-05-10 10:56:27
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,2018-05-07 11:11:27,2018-05-07 11:11:27
2,0000f46a3911fa3c0805444483337064,1,2017-03-10 21:05:03,2017-03-10 21:05:03
3,0000f6ccb0745a6a4b88665a16c9f078,1,2017-10-12 20:29:41,2017-10-12 20:29:41
4,0004aac84e0df4da2b147fca70cf8255,1,2017-11-14 19:45:42,2017-11-14 19:45:42


In [88]:
# ------------------------------------------------------------
# CUSTOMER SPENDING
# ------------------------------------------------------------

order_value = (
    order_items
    .groupby("order_id")
    .agg(
        total_price=("price", "sum"),
        total_freight=("freight_value", "sum"),
        total_items=("order_item_id", "count")
    )
    .reset_index()
)

customer_summary = customer_summary.merge(
    customer_orders[
        ["order_id", "customer_unique_id"]
    ],
    on="customer_unique_id",
    how="left"
)

customer_summary = customer_summary.merge(
    order_value,
    on="order_id",
    how="left"
)

customer_value = (
    customer_summary
    .groupby("customer_unique_id")
    .agg(
        orders=("orders", "first"),
        first_purchase=("first_purchase", "first"),
        last_purchase=("last_purchase", "first"),
        total_price=("total_price", "sum"),
        total_freight=("total_freight", "sum"),
        total_items=("total_items", "sum")
    )
    .reset_index()
)

customer_value["total_spend"] = (
    customer_value["total_price"]
    + customer_value["total_freight"]
)

customer_value["avg_order_value"] = (
    customer_value["total_spend"]
    / customer_value["orders"]
)

customer_value["avg_items_per_order"] = (
    customer_value["total_items"]
    / customer_value["orders"]
)

customer_value.head()

,customer_unique_id,orders,first_purchase,last_purchase,total_price,total_freight,total_items,total_spend,avg_order_value,avg_items_per_order
0,0000366f3b9a7992bf8c76cfdf3221e2,1,2018-05-10 10:56:27,2018-05-10 10:56:27,129.90,12.00,1,141.90,141.90,1.0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,2018-05-07 11:11:27,2018-05-07 11:11:27,18.90,8.29,1,27.19,27.19,1.0
2,0000f46a3911fa3c0805444483337064,1,2017-03-10 21:05:03,2017-03-10 21:05:03,69.00,17.22,1,86.22,86.22,1.0
3,0000f6ccb0745a6a4b88665a16c9f078,1,2017-10-12 20:29:41,2017-10-12 20:29:41,25.99,17.63,1,43.62,43.62,1.0
4,0004aac84e0df4da2b147fca70cf8255,1,2017-11-14 19:45:42,2017-11-14 19:45:42,180.00,16.89,1,196.89,196.89,1.0


In [89]:
# ------------------------------------------------------------
# CUSTOMER PAYMENT BEHAVIOUR
# ------------------------------------------------------------

payment_order = (
    payments
    .groupby("order_id")
    .agg(
        payment_value=("payment_value", "sum"),
        max_installments=("payment_installments", "max")
    )
    .reset_index()
)

customer_payment = (
    customer_orders[
        ["order_id", "customer_unique_id"]
    ]
    .merge(
        payment_order,
        on="order_id",
        how="left"
    )
)

customer_payment = (
    customer_payment
    .groupby("customer_unique_id")
    .agg(
        total_payment=("payment_value", "sum"),
        avg_installments=("max_installments", "mean"),
        max_installments=("max_installments", "max")
    )
    .reset_index()
)

customer_value = customer_value.merge(
    customer_payment,
    on="customer_unique_id",
    how="left"
)

customer_value.head()

,customer_unique_id,orders,first_purchase,last_purchase,total_price,total_freight,total_items,total_spend,avg_order_value,avg_items_per_order,total_payment,avg_installments,max_installments
0,0000366f3b9a7992bf8c76cfdf3221e2,1,2018-05-10 10:56:27,2018-05-10 10:56:27,129.90,12.00,1,141.90,141.90,1.0,141.90,8.0,8.0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,2018-05-07 11:11:27,2018-05-07 11:11:27,18.90,8.29,1,27.19,27.19,1.0,27.19,1.0,1.0
2,0000f46a3911fa3c0805444483337064,1,2017-03-10 21:05:03,2017-03-10 21:05:03,69.00,17.22,1,86.22,86.22,1.0,86.22,8.0,8.0
3,0000f6ccb0745a6a4b88665a16c9f078,1,2017-10-12 20:29:41,2017-10-12 20:29:41,25.99,17.63,1,43.62,43.62,1.0,43.62,4.0,4.0
4,0004aac84e0df4da2b147fca70cf8255,1,2017-11-14 19:45:42,2017-11-14 19:45:42,180.00,16.89,1,196.89,196.89,1.0,196.89,6.0,6.0


In [90]:
# ------------------------------------------------------------
# CUSTOMER REVIEW BEHAVIOUR
# ------------------------------------------------------------

customer_review = (
    reviews
    .merge(
        customer_orders[
            ["order_id", "customer_unique_id"]
        ],
        on="order_id",
        how="inner"
    )
    .groupby("customer_unique_id")
    .agg(
        reviews=("review_score", "count"),
        avg_review_score=("review_score", "mean")
    )
    .reset_index()
)

customer_value = customer_value.merge(
    customer_review,
    on="customer_unique_id",
    how="left"
)

customer_value.head()

,customer_unique_id,orders,first_purchase,last_purchase,total_price,total_freight,total_items,total_spend,avg_order_value,avg_items_per_order,total_payment,avg_installments,max_installments,reviews,avg_review_score
0,0000366f3b9a7992bf8c76cfdf3221e2,1,2018-05-10 10:56:27,2018-05-10 10:56:27,129.90,12.00,1,141.90,141.90,1.0,141.90,8.0,8.0,1.0,5.0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,2018-05-07 11:11:27,2018-05-07 11:11:27,18.90,8.29,1,27.19,27.19,1.0,27.19,1.0,1.0,1.0,4.0
2,0000f46a3911fa3c0805444483337064,1,2017-03-10 21:05:03,2017-03-10 21:05:03,69.00,17.22,1,86.22,86.22,1.0,86.22,8.0,8.0,1.0,3.0
3,0000f6ccb0745a6a4b88665a16c9f078,1,2017-10-12 20:29:41,2017-10-12 20:29:41,25.99,17.63,1,43.62,43.62,1.0,43.62,4.0,4.0,1.0,4.0
4,0004aac84e0df4da2b147fca70cf8255,1,2017-11-14 19:45:42,2017-11-14 19:45:42,180.00,16.89,1,196.89,196.89,1.0,196.89,6.0,6.0,1.0,5.0


In [91]:
# ------------------------------------------------------------
# FINAL CUSTOMER SEGMENTATION DATASET
# ------------------------------------------------------------

customer_segmentation = customer_value[
    [
        "customer_unique_id",
        "orders",
        "total_spend",
        "avg_order_value",
        "total_items",
        "avg_items_per_order",
        "total_freight",
        "avg_installments",
        "max_installments",
        "avg_review_score"
    ]
].copy()

customer_segmentation.head()

,customer_unique_id,orders,total_spend,avg_order_value,total_items,avg_items_per_order,total_freight,avg_installments,max_installments,avg_review_score
0,0000366f3b9a7992bf8c76cfdf3221e2,1,141.90,141.90,1,1.0,12.00,8.0,8.0,5.0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,27.19,27.19,1,1.0,8.29,1.0,1.0,4.0
2,0000f46a3911fa3c0805444483337064,1,86.22,86.22,1,1.0,17.22,8.0,8.0,3.0
3,0000f6ccb0745a6a4b88665a16c9f078,1,43.62,43.62,1,1.0,17.63,4.0,4.0,4.0
4,0004aac84e0df4da2b147fca70cf8255,1,196.89,196.89,1,1.0,16.89,6.0,6.0,5.0


In [93]:
# ============================================================
# PAYMENT DISCREPANCY INVESTIGATION — CLEAN TABLES
# ============================================================

import pandas as pd
import numpy as np


# ============================================================
# 1. FIND ALL PAYMENT DISCREPANCIES
# ============================================================

item_totals = (
    order_items
    .groupby("order_id")
    .agg(
        item_price=("price", "sum"),
        freight=("freight_value", "sum")
    )
    .reset_index()
)

item_totals["item_plus_freight"] = (
    item_totals["item_price"] + item_totals["freight"]
)

payment_totals = (
    payments
    .groupby("order_id")
    .agg(
        total_payment=("payment_value", "sum")
    )
    .reset_index()
)

comparison = item_totals.merge(
    payment_totals,
    on="order_id",
    how="inner"
)

comparison["difference"] = (
    comparison["total_payment"]
    - comparison["item_plus_freight"]
)

comparison["absolute_difference"] = (
    comparison["difference"].abs()
)

discrepancies = comparison[
    comparison["absolute_difference"] > 0.01
].copy()


# ============================================================
# TABLE 1 — BASIC DISCREPANCY SUMMARY
# ============================================================

basic_summary = pd.DataFrame({
    "Metric": [
        "Total discrepant orders",
        "Payment greater than item + freight",
        "Payment lower than item + freight",
        "Average absolute discrepancy",
        "Median absolute discrepancy",
        "Maximum absolute discrepancy"
    ],
    "Value": [
        discrepancies["order_id"].nunique(),
        (discrepancies["difference"] > 0).sum(),
        (discrepancies["difference"] < 0).sum(),
        discrepancies["absolute_difference"].mean(),
        discrepancies["absolute_difference"].median(),
        discrepancies["absolute_difference"].max()
    ]
})

print("\nTABLE 1 — BASIC DISCREPANCY SUMMARY")
display(basic_summary)


# ============================================================
# 2. PAYMENT RECORDS
# ============================================================

payment_structure = (
    payments
    .groupby("order_id")
    .agg(
        payment_records=("payment_sequential", "count")
    )
    .reset_index()
)

payment_structure["payment_structure"] = np.where(
    payment_structure["payment_records"] == 1,
    "Single payment",
    "Multiple payments"
)

discrepancies = discrepancies.merge(
    payment_structure,
    on="order_id",
    how="left"
)


# ============================================================
# TABLE 2 — SINGLE VS MULTIPLE PAYMENT
# ============================================================

payment_record_summary = (
    discrepancies
    .groupby("payment_structure")
    .agg(
        orders=("order_id", "nunique")
    )
    .reset_index()
)

payment_record_summary["percentage"] = (
    payment_record_summary["orders"]
    / len(discrepancies)
    * 100
).round(2)

print("\nTABLE 2 — SINGLE VS MULTIPLE PAYMENT RECORDS")
display(payment_record_summary)


# ============================================================
# 3. PAYMENT TYPE
# ============================================================

payment_type_per_order = (
    payments[
        ["order_id", "payment_type"]
    ]
    .drop_duplicates()
)

discrepancy_payment_types = (
    discrepancies[["order_id"]]
    .merge(
        payment_type_per_order,
        on="order_id",
        how="left"
    )
)


# ============================================================
# TABLE 3 — PAYMENT TYPE
# ============================================================

payment_type_summary = (
    discrepancy_payment_types
    .groupby("payment_type")
    .agg(
        orders=("order_id", "nunique")
    )
    .reset_index()
)

payment_type_summary["percentage"] = (
    payment_type_summary["orders"]
    / len(discrepancies)
    * 100
).round(2)

print("\nTABLE 3 — PAYMENT TYPE AMONG DISCREPANT ORDERS")
display(payment_type_summary)


# ============================================================
# 4. DISCREPANCY SIZE
# ============================================================

discrepancies["difference_band"] = pd.cut(
    discrepancies["absolute_difference"],
    bins=[0, 1, 10, 50, 100, np.inf],
    labels=[
        "$0.01–$1",
        "$1–$10",
        "$10–$50",
        "$50–$100",
        "$100+"
    ]
)


# ============================================================
# TABLE 4 — DISCREPANCY SIZE
# ============================================================

difference_summary = (
    discrepancies
    .groupby("difference_band", observed=True)
    .agg(
        orders=("order_id", "nunique"),
        avg_difference=("absolute_difference", "mean"),
        median_difference=("absolute_difference", "median"),
        maximum_difference=("absolute_difference", "max")
    )
    .reset_index()
)

difference_summary[
    [
        "avg_difference",
        "median_difference",
        "maximum_difference"
    ]
] = difference_summary[
    [
        "avg_difference",
        "median_difference",
        "maximum_difference"
    ]
].round(2)

print("\nTABLE 4 — DISCREPANCY MAGNITUDE")
display(difference_summary)


# ============================================================
# 5. INSTALLMENTS
# ============================================================

installment_summary = (
    payments
    .groupby("order_id")
    .agg(
        max_installments=("payment_installments", "max"),
        avg_installments=("payment_installments", "mean")
    )
    .reset_index()
)

discrepancies = discrepancies.merge(
    installment_summary,
    on="order_id",
    how="left"
)


# ============================================================
# TABLE 5 — INSTALLMENTS AMONG DISCREPANT ORDERS
# ============================================================

installment_table = (
    discrepancies
    .groupby("max_installments")
    .agg(
        orders=("order_id", "nunique"),
        avg_difference=("absolute_difference", "mean"),
        median_difference=("absolute_difference", "median")
    )
    .reset_index()
    .sort_values("max_installments")
)

installment_table[
    ["avg_difference", "median_difference"]
] = installment_table[
    ["avg_difference", "median_difference"]
].round(2)

print("\nTABLE 5 — INSTALLMENTS VS DISCREPANCY")
display(installment_table)


# ============================================================
# 6. PURCHASE VALUE
# ============================================================

discrepancies["purchase_value"] = (
    discrepancies["item_plus_freight"]
)

discrepancies["purchase_band"] = pd.cut(
    discrepancies["purchase_value"],
    bins=[
        0,
        100,
        250,
        500,
        1000,
        2000,
        np.inf
    ],
    labels=[
        "$0–$100",
        "$100–$250",
        "$250–$500",
        "$500–$1,000",
        "$1,000–$2,000",
        "$2,000+"
    ]
)


# ============================================================
# TABLE 6 — PURCHASE VALUE VS DISCREPANCY
# ============================================================

purchase_summary = (
    discrepancies
    .groupby("purchase_band", observed=True)
    .agg(
        orders=("order_id", "nunique"),
        avg_purchase=("purchase_value", "mean"),
        max_purchase=("purchase_value", "max"),
        avg_difference=("absolute_difference", "mean"),
        maximum_difference=("absolute_difference", "max"),
        avg_installments=("avg_installments", "mean")
    )
    .reset_index()
)

purchase_summary[
    [
        "avg_purchase",
        "max_purchase",
        "avg_difference",
        "maximum_difference",
        "avg_installments"
    ]
] = purchase_summary[
    [
        "avg_purchase",
        "max_purchase",
        "avg_difference",
        "maximum_difference",
        "avg_installments"
    ]
].round(2)

print("\nTABLE 6 — PURCHASE VALUE VS DISCREPANCY")
display(purchase_summary)


# ============================================================
# 7. CREDIT CARD + INSTALLMENT CHECK
# ============================================================

credit_installment = (
    discrepancies[["order_id"]]
    .merge(
        payments[
            [
                "order_id",
                "payment_type",
                "payment_installments"
            ]
        ],
        on="order_id",
        how="left"
    )
)


# ============================================================
# TABLE 7 — 1-INSTALLMENT DISCREPANCIES BY PAYMENT TYPE
# ============================================================

one_installment = (
    credit_installment[
        credit_installment["payment_installments"] == 1
    ]
    .groupby("payment_type")
    .agg(
        orders=("order_id", "nunique")
    )
    .reset_index()
)

print("\nTABLE 7 — 1-INSTALLMENT DISCREPANT ORDERS")
display(one_installment)


# ============================================================
# 8. LARGEST DISCREPANCIES
# ============================================================

largest_discrepancies = (
    discrepancies
    .sort_values(
        "absolute_difference",
        ascending=False
    )
    [
        [
            "order_id",
            "item_price",
            "freight",
            "item_plus_freight",
            "total_payment",
            "difference",
            "absolute_difference",
            "payment_records",
            "payment_structure",
            "max_installments"
        ]
    ]
    .head(20)
)


# ============================================================
# TABLE 8 — TOP 20 LARGEST DISCREPANCIES
# ============================================================

print("\nTABLE 8 — TOP 20 LARGEST DISCREPANCIES")
display(largest_discrepancies)


TABLE 1 — BASIC DISCREPANCY SUMMARY


,Metric,Value
0,Total discrepant orders,381.000000
1,Payment greater than item + freight,291.000000
2,Payment lower than item + freight,90.000000
3,Average absolute discrepancy,8.582677
4,Median absolute discrepancy,3.770000
5,Maximum absolute discrepancy,182.810000



TABLE 2 — SINGLE VS MULTIPLE PAYMENT RECORDS


,payment_structure,orders,percentage
0,Multiple payments,21,5.51
1,Single payment,360,94.49



TABLE 3 — PAYMENT TYPE AMONG DISCREPANT ORDERS


...

,purchase_band,orders,avg_purchase,max_purchase,avg_difference,maximum_difference,avg_installments
0,$0–$100,178,58.01,99.43,5.15,19.59,4.50
1,$100–$250,125,158.87,245.90,11.21,61.69,6.77
2,$250–$500,55,324.83,491.53,12.59,119.01,5.25
3,"$500–$1,000",14,669.15,963.36,5.49,76.53,4.50
4,"$1,000–$2,000",8,1425.68,1988.53,22.87,182.81,6.25
5,"$2,000+",1,2913.48,2913.48,0.01,0.01,10.00



TABLE 7 — 1-INSTALLMENT DISCREPANT ORDERS


,payment_type,orders
0,boleto,26
1,credit_card,32
2,debit_card,9
3,voucher,11



TABLE 8 — TOP 20 LARGEST DISCREPANCIES


,order_id,item_price,freight,item_plus_freight,total_payment,difference,absolute_difference,payment_records,payment_structure,max_installments
315,ce6d150fb29ada17d2082f4847107665,1299.00,104.66,1403.66,1586.47,182.81,182.81,1,Single payment,10
167,6e5fe7366a2e1bfbf3257dba0af1267f,179.19,108.72,287.91,406.92,119.01,119.01,1,Single payment,10
174,70b742795bc441e94a44a084b6d9ce7a,269.99,196.94,466.93,578.82,111.89,111.89,1,Single payment,20
236,996c7e73600ad3723e8627ab7bef81e4,559.90,28.00,587.90,664.43,76.53,76.53,1,Single payment,10
175,70b7e94ea46d3e8b5bc12a50186edaf0,167.88,45.27,213.15,274.84,61.69,61.69,1,Single payment,24
285,bc2c82b0ef78d2252b6176d1972db7c9,165.00,77.01,242.01,303.02,61.01,61.01,1,Single payment,21
269,af9ffff2ce6b3defd34fd4c78857a379,395.65,17.52,413.17,466.97,53.80,53.80,1,Single payment,10
54,262118ce178bb3e4590a3adcf6d62e6b,119.80,57.94,177.74,126.12,-51.62,51.62,1,Single payment,5
291,bfdb5bbb06458d600a33d61f5f287472,297.00,51.93,348.93,394.36,45.43,45.43,1,Single payment,10
212,8d9c0dc8d5a2ce804f6b925d8f8e6c1d,209.80,44.65,254.45,293.89,39.44,39.44,1,Single payment,12
